# Public Summary Sources
Generated from public_summary/export_summary.py. No ETL or network execution.

In [0]:
"""Prepare allowlisted reconciliation observations; no network or credential handling."""

from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import json
import os
from pathlib import Path
import re
import tempfile


CHECK_NAMES = frozenset("""
bronze_equals_silver_plus_quarantine bronze_has_rows bronze_null_batch_ids
bronze_split_fare_amount_cents bronze_split_fare_amount_invalid bronze_split_fare_amount_nulls
bronze_split_total_amount_cents bronze_split_total_amount_invalid bronze_split_total_amount_nulls
date_dimension_key_matches_date date_duplicate_key_groups date_null_keys
gold_date_key_matches_pickup_timestamp gold_dropoff_zone_key_matches_original_location
gold_fare_amount_invalid gold_missing_date_key gold_missing_dropoff_zone_key
gold_missing_pickup_zone_key gold_null_batch_ids gold_null_date_key gold_null_dropoff_zone_key
gold_null_pickup_zone_key gold_pickup_date_outside_allowed_window
gold_pickup_zone_key_matches_original_location gold_total_amount_invalid
gold_unexpected_bronze_batch_ids gold_unknown_zone_trips per_batch_layer_metric_mismatches
quarantine_has_error_reasons quarantine_null_batch_ids quarantine_unexpected_bronze_batch_ids
silver_equals_gold_rows silver_fare_amount_invalid silver_gold_fare_amount_cents
silver_gold_fare_amount_invalid silver_gold_fare_amount_nulls silver_gold_total_amount_cents
silver_gold_total_amount_invalid silver_gold_total_amount_nulls silver_null_batch_ids
silver_null_pickup_dates silver_pickup_date_matches_timestamp
silver_pickup_date_outside_allowed_window silver_rows_or_fully_quarantined
silver_total_amount_invalid silver_unexpected_bronze_batch_ids
zone_duplicate_key_groups zone_null_keys silver_pickup_date_outside_source_month
""".split())
WARNING_CHECKS = {'gold_unknown_zone_trips', 'silver_pickup_date_outside_source_month'}
BOOLEAN_CHECKS = {'bronze_has_rows','silver_rows_or_fully_quarantined'}
COMPARISON_CHECKS = {
    'bronze_equals_silver_plus_quarantine','silver_equals_gold_rows',
    'bronze_split_fare_amount_cents','bronze_split_fare_amount_nulls',
    'bronze_split_total_amount_cents','bronze_split_total_amount_nulls',
    'bronze_split_fare_amount_invalid','bronze_split_total_amount_invalid',
    'silver_gold_fare_amount_cents','silver_gold_fare_amount_nulls',
    'silver_gold_total_amount_cents','silver_gold_total_amount_nulls',
}


def _timestamp(value):
    try:
        parsed = datetime.fromisoformat(str(value).replace('Z', '+00:00'))
    except ValueError:
        raise ValueError('Invalid evidence timestamp') from None
    if parsed.tzinfo is None:
        raise ValueError('Evidence timestamps must include a timezone')
    return parsed.astimezone(timezone.utc)


def _integer(value, *, signed=False):
    if isinstance(value, bool):
        raise ValueError('Boolean is not a metric')
    try:
        number = Decimal(str(value))
    except InvalidOperation:
        raise ValueError('Invalid integer metric') from None
    if not number.is_finite() or (number < 0 and not signed) or number != number.to_integral_value():
        raise ValueError('Invalid integer metric or sign')
    if abs(number) > 9007199254740991:
        raise ValueError('Metric exceeds the public client integer range')
    return int(number)


def _months(start, end):
    if not all(re.fullmatch(r'\d{4}-(0[1-9]|1[0-2])', p) for p in (start, end)):
        raise ValueError('Invalid source period')
    first = int(start[:4]) * 12 + int(start[5:]) - 1
    last = int(end[:4]) * 12 + int(end[5:]) - 1
    if not 0 <= last - first < 1200:
        raise ValueError('Invalid coverage range')
    return [f'{n // 12:04d}-{n % 12 + 1:02d}' for n in range(first, last + 1)]


def _same_value(left, right):
    if str(left) in ('True', 'False') or str(right) in ('True', 'False'):
        return str(left) == str(right)
    return _integer(left,signed=True) == _integer(right,signed=True)


def build_snapshot(checks, events, *, policy_version, start, end, generated_at_utc):
    """Reject unsafe evidence before constructing an explicitly non-live snapshot.

    Inputs are internal observations, not arbitrary public requests. Run after ETL
    with no concurrent writer. This does not independently re-scan business tables.
    """
    generated = _timestamp(generated_at_utc)
    periods = _months(start, end)
    if not policy_version:
        raise ValueError('Explicit current policy is required')
    latest = {}
    for event in events:
        period = event['source_period']
        if period not in periods:
            continue
        key = (_timestamp(event['event_time_utc']), str(event.get('event_id', '')))
        if period not in latest or key > latest[period][0]:
            latest[period] = key, event
    grouped = {period: [] for period in periods}
    for check in checks:
        if check['source_period'] in grouped:
            grouped[check['source_period']].append(check)
    months, evidence_times = [], []
    for period in periods:
        if period not in latest or not grouped[period]:
            raise ValueError('Missing source-month evidence')
        event_time, event = latest[period][0][0], latest[period][1]
        if event['policy_version'] != policy_version or event['status'] not in ('COMPLETED','COMPLETED_WARN') or event['stage'] != 'reconciliation':
            raise ValueError('Latest attempt is not a current-policy reconciliation completion')
        rows = grouped[period]
        newest = max(_timestamp(row['reconciled_at_utc']) for row in rows)
        rows = [row for row in rows if _timestamp(row['reconciled_at_utc']) == newest]
        expected_batch = 'nyc_taxi_lakehouse_' + period.replace('-', '_') + '_' + str(event['run_id'])
        if newest > event_time or event_time > generated or any(row['batch_id'] != expected_batch for row in rows):
            raise ValueError('Check evidence does not match the completed attempt')
        if len(rows) != len(CHECK_NAMES) or {row['check_name'] for row in rows} != CHECK_NAMES:
            raise ValueError('Missing, duplicate, or incompatible checks')
        by_name = {row['check_name']: row for row in rows}
        warnings = 0
        for row in rows:
            warning = row['check_name'] in WARNING_CHECKS
            if row['check_name'] in BOOLEAN_CHECKS:
                if str(row['expected_value']) != 'True':
                    raise ValueError('Boolean quality contract changed')
            elif row['check_name'] not in COMPARISON_CHECKS and _integer(row['expected_value']) != 0:
                raise ValueError('Zero-tolerance quality contract changed')
            equal = _same_value(row['expected_value'], row['actual_value'])
            status = 'PASS' if equal else ('WARN' if warning else 'FAIL')
            if row['severity'] != ('WARN' if warning else 'ERROR') or row['check_status'] != status or status == 'FAIL':
                raise ValueError('Failed or inconsistent reconciliation check')
            warnings += status == 'WARN'
        if event['status'] != ('COMPLETED_WARN' if warnings else 'COMPLETED'):
            raise ValueError('Completion status does not match checks')
        def actual(name):
            return _integer(by_name[name]['actual_value'])
        bronze = actual('bronze_equals_silver_plus_quarantine')
        silver = _integer(by_name['silver_equals_gold_rows']['expected_value'])
        gold = actual('silver_equals_gold_rows')
        quarantine = _integer(event['quarantine_count'])
        boundary = actual('silver_pickup_date_outside_source_month')
        if bronze == 0 or bronze != silver + quarantine or silver != gold:
            raise ValueError('Layer counts do not reconcile')
        for key, value in [('bronze_count',bronze),('silver_count',silver),('gold_count',gold),('out_of_period_count',boundary)]:
            if _integer(event[key]) != value:
                raise ValueError('Event and check counts differ')
        months.append(dict(period=period,bronze=bronze,silver=silver,gold=gold,
                           quarantine=quarantine,amountCents=str(actual('silver_gold_total_amount_cents')),
                           boundary=boundary))
        evidence_times.append(newest)
    # Evidence dates must never advance merely because the exporter was rerun.
    return dict(schemaVersion=1,verifiedOn=max(evidence_times).date().isoformat(),
                evidenceDate=max(evidence_times).date().isoformat(),
                evidenceThroughUtc=max(evidence_times).isoformat(),
                generatedAtUtc=generated.isoformat(),publicFeedConnected=False,
                publicationMode='verified_snapshot',months=months)


def write_snapshot(destination, snapshot):
    """Atomically store an already validated local snapshot, never publish it."""
    destination = Path(destination)
    encoded = json.dumps(snapshot,indent=2,allow_nan=False).encode('utf-8')
    destination.parent.mkdir(parents=True,exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=destination.parent,delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(encoded)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary,destination)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()